# small\_fable — inference only (load from Hugging Face)

Read-only. Loads the planner checkpoints trained by `train_all_colab.ipynb` **from the Hugging Face Hub**
and compares, on any prompt you give:

| model | how it answers |
|---|---|
| **Vanilla Qwen-1.5** | base `Qwen/Qwen2.5-1.5B-Instruct`, answers directly (no planner) |
| **Planned Qwen (SFT)** | emits a symbolic plan, then answers conditioned on it |
| **Planned Qwen (SFT+RL)** | same, after off-policy GRPO |

> A GPU helps but isn't required. `Runtime → Change runtime type → T4 GPU` for speed.
> The repo is **public**, so no Hugging Face token is needed to download.


## 1 · Setup — clone repo (for the model code) + install

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
%cd small_fable
!pip install -q transformers peft accelerate safetensors huggingface_hub
print('ready')


## 2 · Config — which HF repo + the prompt
Set `HF_REPO` to the value the training notebook printed (`<your-username>/small_fable-planner`).
Edit `PROMPT` to whatever you want to test.

In [ ]:
HF_REPO = 'CHANGE_ME/small_fable-planner'   # <-- the repo the training notebook printed
DEVICE  = 'cuda'                            # 'cpu' also works (slower)
TEMP    = 0.7
MAX_NEW = 96

PROMPT = ('A gardener plants 3 rows of 7 tulips and 2 rows of 5 roses. '
          'If 4 tulips wilt, how many flowers are left?')
print('prompt:', PROMPT)


## 3 · Download the checkpoints from Hugging Face
Pulls `joint_ckpt/` (SFT) and `rl_ckpt/` (SFT+RL) into a local folder.

In [ ]:
from huggingface_hub import snapshot_download
local = snapshot_download(repo_id=HF_REPO, repo_type='model',
                          allow_patterns=['joint_ckpt/*', 'rl_ckpt/*'])
SFT_CKPT = os.path.join(local, 'joint_ckpt')
RL_CKPT  = os.path.join(local, 'rl_ckpt')
print('downloaded to:', local)
print('  SFT:', os.path.isdir(SFT_CKPT), '| RL:', os.path.isdir(RL_CKPT))


## 4 · Load the three models
Vanilla Qwen via plain `transformers`; the two planner models via `JointModel.from_checkpoint`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from model_joint import JointModel, decode_plan

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
dtype = torch.float32 if DEVICE == 'cpu' else 'auto'

# vanilla base model
base_tok = AutoTokenizer.from_pretrained(BASE)
base_lm  = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=dtype).to(DEVICE).eval()

# planner models (SFT, SFT+RL)
sft = JointModel.from_checkpoint(BASE, SFT_CKPT, device=DEVICE); sft.eval()
rl  = JointModel.from_checkpoint(BASE, RL_CKPT,  device=DEVICE); rl.eval()
print('models loaded')


## 5 · Generate & compare

In [ ]:
@torch.no_grad()
def vanilla_answer(prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    text = base_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = base_tok(text, return_tensors='pt').to(DEVICE)
    out = base_lm.generate(**ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP,
                           top_p=0.95, pad_token_id=base_tok.eos_token_id)
    return base_tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def planner_answer(m, prompt):
    p_ids, p_attn = m.batch_prompts([prompt])
    plan = m.sample_plan(p_ids, p_attn, temp=TEMP, sample=True)
    gen  = m.generate_answer(p_ids, p_attn, plan, temp=TEMP, sample=True, max_new_tokens=MAX_NEW)
    return ' → '.join(decode_plan(plan[0])), m.tok.decode(gen[0], skip_special_tokens=True).strip()

def rule(t): print('\n' + '='*70 + f'\n{t}\n' + '='*70)
print('PROMPT:', PROMPT)

rule('1) Vanilla Qwen-1.5  (no planner)')
print(vanilla_answer(PROMPT))

rule('2) Planned Qwen-1.5  (SFT)')
plan, ans = planner_answer(sft, PROMPT)
print('plan  :', plan); print('answer:', ans)

rule('3) Planned Qwen-1.5  (SFT + GRPO)')
plan, ans = planner_answer(rl, PROMPT)
print('plan  :', plan); print('answer:', ans)
